In [ ]:
import pandas as pd
from pathlib import Path

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

root = _find_root("out")
primary_path = root / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
fallback_path = root / "out" / "filtered" / "accidents_model_ready_kept.csv"
path = primary_path if primary_path.exists() else fallback_path
assert path.exists(), f"CSV introuvable: {primary_path} ou {fallback_path}"
print(f"Dataset utilise: {path}")

# Lecture légère (5 lignes) pour lister les colonnes dispo
df = pd.read_csv(path, sep=";", nrows=5)
time_cols = [c for c in ["hrmn", "hour", "minute", "time_bucket"] if c in df.columns]
print("Colonnes temps trouvées:", time_cols)

# Vérif rapide des bornes (échantillon) uniquement si les colonnes existent
if time_cols:
    df2 = pd.read_csv(
        path,
        sep=";",
        usecols=time_cols,
        nrows=5000,
        low_memory=False,
    )

    if "hour" in df2.columns:
        print("hour min/max:", df2["hour"].min(), df2["hour"].max())
    if "minute" in df2.columns:
        print("minute min/max:", df2["minute"].min(), df2["minute"].max())
    if "hrmn" in df2.columns:
        print("exemples hrmn:", df2["hrmn"].head().tolist())
else:
    print("Aucune colonne temps trouvée (hrmn/hour/minute).")


## taux de gravité par heure 

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

# --- paramètres ---
def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

root = _find_root("out")
primary_path = root / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
fallback_path = root / "out" / "filtered" / "accidents_model_ready_kept.csv"
path = primary_path if primary_path.exists() else fallback_path
assert path.exists(), f"CSV introuvable: {primary_path} ou {fallback_path}"
print(f"Dataset utilise: {path}")

SEP = ";"  # ton CSV
TARGET = "grave"  # colonne cible (0/1)

# vérifie que la cible existe
cols = pd.read_csv(path, sep=SEP, nrows=0).columns
if TARGET not in cols:
    raise ValueError(f"Colonne cible introuvable: {TARGET}. Colonnes dispo: {list(cols)[:10]}...")

# --- lecture minimale ---
usecols = [c for c in ["hour", TARGET] if c is not None]
df = pd.read_csv(path, sep=SEP, usecols=usecols)

# sécurité
df = df.dropna(subset=["hour", TARGET]).copy()
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")
df = df.dropna(subset=["hour"])
df["hour"] = df["hour"].astype(int)

# groupby : taux de grave + volume
g = (df.groupby("hour")[TARGET]
       .agg(rate="mean", n="size")
       .reset_index()
       .sort_values("hour"))

# conversion en %
g["rate_pct"] = 100 * g["rate"]

display(g)

fig = px.line(
    g, x="hour", y="rate_pct",
    markers=True,
    title="Taux d'accidents graves (y=1) par heure",
    labels={"hour": "Heure (0–23)", "rate_pct": "Taux de grave (%)"}
)
fig.show()

fig2 = px.bar(
    g, x="hour", y="n",
    title="Volume d'accidents par heure",
    labels={"hour": "Heure (0–23)", "n": "Nombre de cas"}
)
fig2.show()


## creation d’un time bucket pour la gesiton de l’heure
night : 00–05
morning : 06–11
afternoon : 12–17
evening : 18–23


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

root = _find_root("out")
primary_path = root / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
fallback_path = root / "out" / "filtered" / "accidents_model_ready_kept.csv"
path = primary_path if primary_path.exists() else fallback_path
assert path.exists(), f"CSV introuvable: {primary_path} ou {fallback_path}"
print(f"Dataset utilise: {path}")

SEP = ";"
TARGET = "grave"  # colonne cible (0/1)

# vérifie que la cible existe
cols = pd.read_csv(path, sep=SEP, nrows=0).columns
if TARGET not in cols:
    raise ValueError(f"Colonne cible introuvable: {TARGET}. Colonnes dispo: {list(cols)[:10]}...")

df = pd.read_csv(path, sep=SEP, usecols=["hour", TARGET]).dropna(subset=["hour", TARGET]).copy()
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")
df = df.dropna(subset=["hour"])
df["hour"] = df["hour"].astype(int)

def hour_to_bucket(h: int) -> str:
    if 0 <= h <= 5:
        return "night_00_05"
    elif 6 <= h <= 11:
        return "morning_06_11"
    elif 12 <= h <= 17:
        return "afternoon_12_17"
    elif 18 <= h <= 23:
        return "evening_18_23"
    return "unknown"

df["time_bucket"] = df["hour"].apply(hour_to_bucket)

bucket_order = ["night_00_05", "morning_06_11", "afternoon_12_17", "evening_18_23", "unknown"]

g = (df.groupby("time_bucket")[TARGET]
       .agg(rate="mean", n="size")
       .reindex(bucket_order)
       .reset_index())

g["rate_pct"] = 100 * g["rate"]
display(g)

fig = px.bar(
    g, x="time_bucket", y="rate_pct", text="rate_pct",
    title="Taux d'accidents graves (y=1) par tranche horaire (time_bucket)",
    labels={"time_bucket": "Tranche horaire", "rate_pct": "Taux de grave (%)"}
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(xaxis={"categoryorder": "array", "categoryarray": bucket_order})
fig.show()

fig2 = px.bar(
    g, x="time_bucket", y="n", text="n",
    title="Volume d'accidents par tranche horaire",
    labels={"time_bucket": "Tranche horaire", "n": "Nombre de cas"}
)
fig2.update_layout(xaxis={"categoryorder": "array", "categoryarray": bucket_order})
fig2.show()


## creation de la cat time bucker  pour maj du dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

root = _find_root("out")
path_in = root / "out" / "filtered" / "accidents_model_ready_kept.csv"
SEP = ";"
TARGET = "grave"  # adapte si besoin

df = pd.read_csv(path_in, sep=SEP)

# time_bucket depuis hour
df["hour"] = pd.to_numeric(df["hour"], errors="coerce")

def hour_to_bucket(h):
    if pd.isna(h): return "unknown"
    h = int(h)
    if 0 <= h <= 5:   return "night_00_05"
    if 6 <= h <= 11:  return "morning_06_11"
    if 12 <= h <= 17: return "afternoon_12_17"
    if 18 <= h <= 23: return "evening_18_23"
    return "unknown"

df["time_bucket"] = df["hour"].apply(hour_to_bucket)

# (option) supprimer minute si tu veux la V2
# df = df.drop(columns=["minute"], errors="ignore")

out_dir = root / "out"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "accidents_model_ready_kept_with_time_bucket.csv"
df.to_csv(out_csv, sep=";", index=False)

print("✅ écrit:", out_csv)
print(df["time_bucket"].value_counts(dropna=False))


## v2 integration de time_bucket dans top 15 feature

In [2]:
product15_v2 = [
    "dep",
    "lum",
    "atm",
    "catr",
    "agg",
    "int",
    "circ",
    "col",
    "vma_bucket",
    "catv_family_4",
    "manv_mode",
    "driver_age_bucket",
    "choc_mode",
    "driver_trajet_family",
    "time_bucket",
]


## controle et creation X, y

In [3]:
import pandas as pd
from pathlib import Path

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

TARGET = "grave"

# Rechargement explicite depuis le CSV complet — independant du df global
# (les cellules d'analyse precedentes ecrasent df avec seulement 2 colonnes)
_csv = _find_root("out") / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
assert _csv.exists(), f"CSV introuvable: {_csv} — lance d'abord la cellule 93d9a19a"
df = pd.read_csv(_csv, sep=";")

missing = [c for c in product15_v2 + [TARGET] if c not in df.columns]
assert not missing, f"Colonnes manquantes: {missing}"

X = df[product15_v2].copy()
y = df[TARGET].astype(int).copy()

print("✅ X shape:", X.shape)
print("✅ y mean (positifs):", y.mean())
print("✅ Valeurs time_bucket:", X["time_bucket"].value_counts())


✅ X shape: (164526, 15)
✅ y mean (positifs): 0.36075757023206056
✅ Valeurs time_bucket: time_bucket
afternoon_12_17    60687
evening_18_23      44732
morning_06_11      44358
night_00_05        14749
Name: count, dtype: int64


## gestion des features en string

In [4]:
MISSING_CAT = "__MISSING__"
cat_cols = product15_v2[:]  # les 15 sont catégorielles dans ton design produit

for c in cat_cols:
    X[c] = X[c].astype("string").fillna(MISSING_CAT).astype(str)

print("✅ Types après cast:")
print(X.dtypes.value_counts())


✅ Types après cast:
object    15
Name: count, dtype: int64


## Pool CatBoost + baseline CV 5 folds

from catboost import Pool, cv

train_pool = Pool(X, y, cat_features=cat_cols)

baseline_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 4000,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 3.0,
    "random_seed": 42,
    "verbose": False
}

cv_res = cv(
    pool=train_pool,
    params=baseline_params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42,
    early_stopping_rounds=200,
    verbose=200
)

best_idx = int(cv_res["test-AUC-mean"].idxmax())
best_auc = float(cv_res.loc[best_idx, "test-AUC-mean"])
best_iter = int(cv_res.loc[best_idx, "iterations"])
print("✅ Baseline CV AUC:", best_auc, "| best_iter:", best_iter)


## optuna  + catboost CV 5folds

import optuna
from catboost import Pool, cv

train_pool = Pool(X, y, cat_features=cat_cols)

RANDOM_SEED = 42
N_TRIALS = 30      # monte à 60 si tu veux plus fort
MAX_ITERS = 4000
EARLY_STOP = 200
FOLDS = 5

def objective(trial: optuna.Trial) -> float:
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": MAX_ITERS,
        "random_seed": RANDOM_SEED,
        "verbose": False,

        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 50.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.5),
        "border_count": trial.suggest_int("border_count", 64, 255),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
    }

    cv_res = cv(
        pool=train_pool,
        params=params,
        fold_count=FOLDS,
        shuffle=True,
        partition_random_seed=RANDOM_SEED,
        early_stopping_rounds=EARLY_STOP,
        verbose=False
    )

    return float(cv_res["test-AUC-mean"].max())

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("✅ Best CV AUC:", study.best_value)
print("✅ Best params:", study.best_params)


## optuna plus rapide

import optuna
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

RANDOM_SEED = 42
N_TRIALS = 5          # fais 10-20, ça ira vite
MAX_ITERS = 6000
EARLY_STOP = 200

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

def objective(trial):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": MAX_ITERS,
        "random_seed": RANDOM_SEED,
        "verbose": 0,
        "od_type": "Iter",
        "od_wait": EARLY_STOP,

        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 50.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.5),
        "border_count": trial.suggest_int("border_count", 64, 255),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
    }

    m = CatBoostClassifier(**params)
    m.fit(
        X_train, y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    proba = m.predict_proba(X_valid)[:, 1]
    return roc_auc_score(y_valid, proba)

study_fast = optuna.create_study(direction="maximize")
study_fast.optimize(objective, n_trials=N_TRIALS)

print("✅ Best holdout AUC:", study_fast.best_value)
print("✅ Best params:", study_fast.best_params)


## Validation 1 tial CV 5 fold

In [4]:
from catboost import Pool, cv

trial1_params = {
    "depth": 5,
    "learning_rate": 0.023227316785394195,
    "l2_leaf_reg": 0.055467338800778004,
    "random_strength": 5.953804363406876,
    "bagging_temperature": 1.10434276499118,
    "border_count": 114,
    "subsample": 0.9360904372035957,
    "rsm": 0.8799642206944849,
}

train_pool = Pool(X, y, cat_features=cat_cols)

params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 200,
    "random_seed": 42,
    "verbose": False,
    **trial1_params
}

cv_res = cv(
    pool=train_pool,
    params=params,
    fold_count=3,
    shuffle=True,
    partition_random_seed=42,
    early_stopping_rounds=200,
    verbose=200
)

best_idx = int(cv_res["test-AUC-mean"].idxmax())
best_auc = float(cv_res.loc[best_idx, "test-AUC-mean"])
best_iter = int(cv_res.loc[best_idx, "iterations"])

print("✅ Trial1 confirmed CV AUC:", best_auc, "| best_iter:", best_iter)


Training on fold [0/5]
0:	test: 0.7702919	best: 0.7702919 (0)	total: 163ms	remaining: 16m 19s
200:	test: 0.8097177	best: 0.8097177 (200)	total: 14.2s	remaining: 6m 48s
400:	test: 0.8127072	best: 0.8127072 (400)	total: 34.5s	remaining: 8m 1s
600:	test: 0.8164130	best: 0.8164130 (600)	total: 50.1s	remaining: 7m 30s
800:	test: 0.8191908	best: 0.8191908 (800)	total: 1m 6s	remaining: 7m 10s
1000:	test: 0.8204508	best: 0.8204508 (1000)	total: 1m 22s	remaining: 6m 52s
1200:	test: 0.8211934	best: 0.8211934 (1200)	total: 1m 37s	remaining: 6m 29s
1400:	test: 0.8217598	best: 0.8217598 (1400)	total: 1m 52s	remaining: 6m 9s
1600:	test: 0.8220956	best: 0.8220963 (1599)	total: 2m 7s	remaining: 5m 49s
1800:	test: 0.8223930	best: 0.8223966 (1799)	total: 2m 22s	remaining: 5m 32s
2000:	test: 0.8226463	best: 0.8226463 (2000)	total: 2m 39s	remaining: 5m 19s
2200:	test: 0.8228079	best: 0.8228110 (2198)	total: 2m 54s	remaining: 5m 1s
2400:	test: 0.8229446	best: 0.8229458 (2395)	total: 3m 10s	remaining: 4m 44

## best export 

## entrainement final + export 

In [ ]:
from catboost import CatBoostClassifier
from pathlib import Path
import json, datetime

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

out_dir = _find_root("out") / "out"
out_dir.mkdir(parents=True, exist_ok=True)

model_path = out_dir / "catboost_product15_v2_time_bucket_final.cbm"
meta_path  = out_dir / "catboost_product15_v2_time_bucket_final_meta.json"

final_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    iterations=best_iter,
    verbose=200,
    **trial1_params
)

final_model.fit(X, y, cat_features=cat_cols)
final_model.save_model(str(model_path))

meta = {
    "model_name": "catboost_product15_v2_time_bucket_final",
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "threshold": 0.47,
    "features": product15_v2,
    "cat_features": cat_cols,
    "catboost_params": {**trial1_params, "iterations": best_iter},
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("✅ Model:", model_path)
print("✅ Meta:", meta_path)


## Integration MLflow (CatBoost time bucket)

Bloc pret pour la prochaine etape. Par defaut, MLflow est desactive.

In [5]:
from sklearn.model_selection import train_test_split

# Holdout 80/20 — meme seed que l'entrainement final pour reproductibilite
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape} | positifs : {y_train.mean():.3f}")
print(f"Valid : {X_valid.shape} | positifs : {y_valid.mean():.3f}")


Train : (131620, 15) | positifs : 0.361
Valid : (32906, 15) | positifs : 0.361


In [6]:
ENABLE_MLFLOW = False
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "accidentologie_model_benchmark"
ENABLE_MODEL_REGISTRY = True
REGISTERED_MODEL_NAME = "briefml-catboost-product15-v2-time-bucket"

if ENABLE_MLFLOW:
    from pathlib import Path
    import json as _json

    import numpy as np
    import mlflow
    import mlflow.catboost
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    def _find_root(marker="out") -> Path:
        for p in [Path.cwd(), *Path.cwd().parents]:
            if (p / marker).exists():
                return p
        raise RuntimeError("Racine projet introuvable: dossier 'out' absent depuis le cwd courant.")

    def _to_params(d):
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(v, (int, float, str, bool, np.integer, np.floating, np.bool_)):
                    out[f"catboost_{k}"] = v.item() if hasattr(v, "item") else v
        return out

    def _binary_metrics(y_true, proba, threshold=0.5, prefix="valid_"):
        pred = (proba >= threshold).astype(int)
        return {
            f"{prefix}threshold": float(threshold),
            f"{prefix}accuracy": float(accuracy_score(y_true, pred)),
            f"{prefix}precision": float(precision_score(y_true, pred, zero_division=0)),
            f"{prefix}recall": float(recall_score(y_true, pred, zero_division=0)),
            f"{prefix}f1": float(f1_score(y_true, pred, zero_division=0)),
            f"{prefix}roc_auc": float(roc_auc_score(y_true, proba)),
            f"{prefix}pr_auc": float(average_precision_score(y_true, proba)),
        }

    root = _find_root("out")
    out_dir = root / "out"

    model_file = out_dir / "catboost_product15_v2_time_bucket_final.cbm"
    meta_file = out_dir / "catboost_product15_v2_time_bucket_final_meta.json"
    model_obj = globals().get("final_model")
    registered_version = None

    if model_obj is None and model_file.exists():
        from catboost import CatBoostClassifier
        model_obj = CatBoostClassifier()
        model_obj.load_model(str(model_file))

    if model_obj is None and not model_file.exists():
        raise RuntimeError("Aucun modele final trouve (objet final_model ou fichier .cbm).")

    # --- threshold : globals > meta.json sur disque > 0.5 ---
    threshold_default = 0.5
    if isinstance(globals().get("meta"), dict):
        t = meta.get("threshold")
        if isinstance(t, (int, float)):
            threshold_default = float(t)
    elif meta_file.exists():
        with open(meta_file, encoding="utf-8") as _f:
            _meta_disk = _json.load(_f)
        t = _meta_disk.get("threshold")
        if isinstance(t, (int, float)):
            threshold_default = float(t)
            print(f"[mlflow] threshold lu depuis meta.json: {threshold_default}")

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="catboost_product15_v2_time_bucket_final"):
        tags = {
            "notebook": "11catboost_check_time_columns.ipynb",
            "model_family": "catboost",
            "model_flavor": "catboost",
            "tag": "accidentologie",
        }
        mlflow.set_tags(tags)
        mlflow.set_tag("registry_enabled", str(ENABLE_MODEL_REGISTRY).lower())

        mlflow.log_param("optimized_for", "auc")
        mlflow.log_param("cv_primary_metric", "auc")

        if isinstance(globals().get("trial1_params"), dict):
            p = _to_params(trial1_params)
            if p:
                mlflow.log_params(p)

        if "best_iter" in globals():
            mlflow.log_param("best_iter", int(best_iter))

        if "best_auc" in globals():
            mlflow.log_metric("cv_primary_score", float(best_auc))

        if not all(k in globals() for k in ["X_valid", "y_valid"]):
            raise RuntimeError(
                "Holdout obligatoire manquant: defini X_valid et y_valid avant le logging MLflow. "
                "Aucun fallback train_proxy n'est autorise."
            )

        eval_split = "valid_holdout"
        eval_X = X_valid
        eval_y = y_valid

        if "X" in globals() and eval_X is X:
            raise RuntimeError("X_valid ne doit pas etre le meme objet que X (train).")

        if model_obj is not None:
            try:
                proba_eval = model_obj.predict_proba(eval_X)[:, 1]
                eval_metrics = _binary_metrics(eval_y, proba_eval, threshold_default, prefix="valid_")
                mlflow.log_metrics(eval_metrics)
                mlflow.set_tag("evaluation_split", eval_split)
            except Exception as exc:
                print(f"[mlflow] evaluation metrics warning: {exc}")

        if model_obj is not None:
            model_info = mlflow.catboost.log_model(
                model_obj,
                artifact_path="model",
                registered_model_name=REGISTERED_MODEL_NAME if ENABLE_MODEL_REGISTRY else None,
            )
            if ENABLE_MODEL_REGISTRY and getattr(model_info, "registered_model_version", None):
                registered_version = model_info.registered_model_version

        if model_file.exists():
            mlflow.log_artifact(str(model_file), artifact_path="model_files")

        if meta_file.exists():
            mlflow.log_artifact(str(meta_file), artifact_path="metadata")

    print("[mlflow] run logged: catboost_product15_v2_time_bucket_final")
    if ENABLE_MODEL_REGISTRY:
        if registered_version is not None:
            print(f"[mlflow] model registered: {REGISTERED_MODEL_NAME} v{registered_version}")
        else:
            print("[mlflow] model registry enabled, but no version was returned.")
else:
    print("MLflow desactive. Passe ENABLE_MLFLOW=True quand le serveur sera pret.")


[mlflow] threshold lu depuis meta.json: 0.47


2026/02/23 19:47:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'briefml-catboost-product15-v2-time-bucket'.
2026/02/23 19:47:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: briefml-catboost-product15-v2-time-bucket, version 1
Created version '1' of model 'briefml-catboost-product15-v2-time-bucket'.


🏃 View run catboost_product15_v2_time_bucket_final at: http://127.0.0.1:5000/#/experiments/2/runs/41e662fd88684facb78fa046f377af8b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[mlflow] run logged: catboost_product15_v2_time_bucket_final
[mlflow] model registered: briefml-catboost-product15-v2-time-bucket v1
